# ZeroTuition — Google Colab Edition

Run the ZeroTuition coupon scanner + auto-enroller in the cloud, no installation needed.

**How to use:** run the cells below in order, top to bottom.

1. **Setup** — clones the repo and installs dependencies.
2. **Preferences** — pick languages / categories / sites (edit the `True`/`False` flags).
3. **Login** — enter your Udemy email + password (hidden input).
4. **Scan & Enroll** — scrapes coupon sites and enrolls you in matching courses.
5. **Results** — shows and downloads your enrolled-courses list.

**Please note:**
- You are entering your Udemy credentials on a Google-hosted machine. Use an account you are comfortable with.
- Automating enrollment may violate Udemy's Terms of Service — use at your own risk.
- Colab runs on datacenter IPs; if Udemy or a coupon site throttles it, retry later or run the desktop app locally.
- Keep the browser tab open while it runs — free Colab runtimes reset when idle or after a few hours.
- This is a fork of techtanic's Discounted-Udemy-Course-Enroller (AGPL-3.0).

In [ ]:
#@title 1. Setup — clone the repo and install dependencies
import os
if not os.path.isdir("zero-tuition"):
    !git clone -q https://github.com/stewardobeng/zero-tuition.git
%cd zero-tuition
!pip install -q -r requirements.txt
print("Setup complete")

In [ ]:
#@title 2. Preferences — edit the True/False flags, then run
import json

languages = {
    "Arabic": False, "Chinese": False, "Dutch": False, "English": True,
    "French": False, "German": False, "Hindi": False, "Indonesian": False,
    "Italian": False, "Japanese": False, "Korean": False, "Nepali": False,
    "Polish": False, "Portuguese": False, "Romanian": False, "Russian": False,
    "Spanish": False, "Thai": False, "Turkish": False, "Urdu": False,
    "Vietnamese": False,
}

categories = {
    "Business": True, "Design": True, "Development": True,
    "Finance & Accounting": True, "Health & Fitness": True,
    "IT & Software": True, "Lifestyle": True, "Marketing": True,
    "Music": True, "Office Productivity": True, "Personal Development": True,
    "Photography & Video": True, "Teaching & Academics": True,
}

sites = {
    "Real Discount": True, "Courson": True, "IDownloadCoupons": True,
    "E-next": True, "Discudemy": True, "Udemy Freebies": True,
    "Course Joiner": True, "Course Vania": True,
}

settings = {
    "sites": sites,
    "languages": languages,
    "categories": categories,
    "min_rating": 0.0,
    "title_exclude": [],
    "instructor_exclude": [],
    "email": "",
    "password": "",
    "save_txt": True,
    "discounted_only": False,
    "use_browser_cookies": False,
    "course_update_threshold_months": 24,
}

with open("zerotuition-cli-settings.json", "w") as f:
    json.dump(settings, f, indent=4)

print("Preferences saved:", sum(languages.values()), "languages,",
      sum(categories.values()), "categories,",
      sum(sites.values()), "sites")

In [ ]:
#@title 3. Login with your Udemy account
from getpass import getpass
from base import Udemy, LoginException

udemy = Udemy("cli")
udemy.load_settings()

email = input("Udemy email: ")
password = getpass("Udemy password: ")

try:
    udemy.manual_login(email, password)
    udemy.get_session_info()
    print(f"Logged in as: {udemy.display_name}")
except LoginException as e:
    print("Login failed:", e)
    print("If Udemy mentions 'too many logins', wait about an hour and retry.")
    raise

In [ ]:
#@title 4. Scan coupon sites and enroll — takes several minutes
import threading
import time
from base import Scraper, scraper_dict

udemy.update_progress = lambda: None  # no live UI inside a notebook
if udemy.is_user_dumb():
    raise RuntimeError(
        "Pick at least one site, language and category in the preferences cell."
    )

scraper = Scraper(udemy.sites)

def scan_site(site):
    code = scraper_dict[site]
    threading.Thread(target=getattr(scraper, code), daemon=True).start()
    while getattr(scraper, f"{code}_length") == 0 and not getattr(
        scraper, f"{code}_error"
    ):
        time.sleep(0.3)
    while not getattr(scraper, f"{code}_done") and not getattr(
        scraper, f"{code}_error"
    ):
        time.sleep(0.5)
    print(f"  {site}: {len(getattr(scraper, f'{code}_data', []))} coupons")

print("Scanning coupon sites...")
udemy.scraped_data = scraper.get_scraped_courses(scan_site)
print(f"Found {len(udemy.scraped_data)} unique courses. Enrolling...\n")

udemy.start_new_enroll()

print("\n=== RESULTS ===")
print("Successfully Enrolled:", udemy.successfully_enrolled_c)
print("Already Enrolled:", udemy.already_enrolled_c)
print("Expired Coupons:", udemy.expired_c)
print("Excluded (filters):", udemy.excluded_c)
print("Failed:", udemy.failed_c)
print("Amount Saved:", round(udemy.amount_saved_c, 2), udemy.currency.upper())

In [ ]:
#@title 5. Show and download your enrolled-courses list
import glob

files = sorted(glob.glob("Courses/*.txt"))
if files:
    latest = files[-1]
    print("Enrolled list:", latest)
    print("-" * 60)
    print(open(latest, encoding="utf-8").read()[:2000])
    try:
        from google.colab import files as colab_files
        colab_files.download(latest)
    except Exception:
        pass  # not running inside Colab
else:
    print("No courses file yet - run the previous cell first.")